<a href="https://colab.research.google.com/github/kny1209/test2/blob/main/AI/DR-08079_%EA%B5%AC%EB%82%98%EC%98%81_AI%EC%9D%91%EC%9A%A9_4%EC%B0%A8%EC%8B%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

문항 1. MeanShift와 CamShift의 차이점 MeanShift 알고리즘은 데이터의 밀도가 가장 높은 곳으로 윈도우를 이동시키며 객체를 추적하지만, 윈도우의 크기가 고정되어 있다는 단점이 있습니다. 이를 개선하여 객체의 크기와 회전 변화에 따라 윈도우의 크기와 방향을 동적으로 조절할 수 있게 만든 알고리즘의 이름은 무엇입니까?

답변 작성란: CamShift 알고리즘

문항 2. 템플릿 매칭(Template Matching) 작은 이미지(템플릿)를 큰 이미지(원본) 위에서 이동시키며 가장 유사한 부분을 찾는 기법을 템플릿 매칭이라고 합니다. OpenCV의 cv2.matchTemplate 함수를 사용하면 결과로 **유사도 맵(2차원 배열)** 이 나옵니다. 이 결과 행렬에서 최댓값(가장 유사한 위치)의 좌표를 찾기 위해 사용하는 OpenCV 함수는 무엇입니까?

답변 작성란: cv2.minMaxLoc()

문항 3. 배경 제거 (Background Subtraction) 구현 CCTV 영상 등에서 고정된 배경을 제거하고 움직이는 객체만 추출하기 위해 배경 제거 알고리즘을 사용합니다. 제공된 실습 파일(background.py)을 참고하여 MOG(Mixture of Gaussians) 알고리즘 객체를 생성하고, 마스크를 추출하는 코드를 작성하세요.

In [ ]:
import cv2
import numpy as np

cap = cv2.VideoCapture('newyork.mp4') # 실습용 비디오 파일

# TODO: MOG 배경 제거 객체 생성 (cv2.bgsegm 모듈 사용)
# (OpenCV 버전에 따라 cv2.createBackgroundSubtractorMOG2()를 쓰기도 하지만,
# 강의 코드인 cv2.bgsegm.createBackgroundSubtractorMOG()를 사용하세요)
fgbg = cv2.bgsegm.createBackgroundSubtractorMOG()

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    # TODO: 프레임에 배경 제거 적용하여 마스크(fgmask) 생성
    fgmask = fgbg.apply(frame)

    # cv2.imshow('Original', frame)
    # cv2.imshow('MOG Mask', fgmask)
    if cv2.waitKey(30) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

문항 4. 템플릿 매칭 (Template Matching) game.jpg 화면에서 coin.png 아이템의 위치를 찾으려 합니다. cv2.matchTemplate을 사용하여 매칭을 수행하고, 가장 유사도가 높은 위치(Top-Left)를 찾는 코드를 완성하세요.

(매칭 메서드는 cv2.TM_CCOEFF_NORMED를 사용합니다.)

In [ ]:
img = cv2.imread('game.jpg')
template = cv2.imread('coin.png')

# TODO: 템플릿 매칭 수행
res = cv2.matchTemplate(img, template, cv2.TM_CCOEFF_NORMED)

# TODO: 매칭 결과에서 최소/최대 값과 위치 찾기
min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(res)

# TM_CCOEFF_NORMED는 최대값이 가장 유사한 지점임
top_left = max_loc
print(f"Detected Location: {top_left}")

문항 5. 추적을 위한 히스토그램 역투영 (Backprojection) MeanShift나 CamShift를 사용하기 위해서는 먼저 추적할 대상(ROI)의 색상 히스토그램을 구하고, 전체 영상에서 해당 색상 분포를 가진 영역을 확률로 표현하는 역투영(Backprojection) 과정이 필요합니다. 아래 코드를 완성하세요.

In [ ]:
# roi: 추적할 객체 이미지 (HSV 색공간)
# frame_hsv: 현재 프레임 (HSV 색공간)

# 1. ROI의 히스토그램 계산 (Hue 채널만 사용: [0], 범위: [0, 180])
roi_hist = cv2.calcHist([roi], [0], None, [180], [0, 180])

# 2. 히스토그램 정규화 (0~255 범위로)
cv2.normalize(roi_hist, roi_hist, 0, 255, cv2.NORM_MINMAX)

# TODO: 현재 프레임(frame_hsv)에 히스토그램 역투영 적용
# 인자: [이미지], [채널], 히스토그램, [범위], scale
dst = cv2.calcBackProject([frame_hsv], [0], roi_hist, [0,180],1)

문항 6. Lucas-Kanade Optical Flow 이전 프레임의 코너점(prev_pts)들이 현재 프레임(curr_frame)에서 어디로 이동했는지 추적하려 합니다. OpenCV의 피라미드 기반 루카스-카나데(Lucas-Kanade) 함수를 사용하여 next_pts를 구하는 코드를 작성하세요.

In [ ]:
# prev_gray: 이전 프레임 (Gray)
# gray: 현재 프레임 (Gray)
# prev_pts: 추적할 특징점들 (numpy array)

lk_params = dict(winSize=(15, 15), maxLevel=2,
                 criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

# TODO: Lucas-Kanade Optical Flow 계산
# 반환값: next_pts(이동한 좌표), status(성공여부), err(오차)
next_pts, status, err = cv2.calcOpicalFlowPyrLK(prev_gray, gray, prev_pts, None, criteria = termcriteria)